# UJIAN TENGAH SEMESTER MACHINE LEARNING

## Prediksi Pembatalan Pemesanan Hotel Menggunakan Algoritma K-Nearest Neighbors (KNN)

**Nama:** Bernardino brian saputra soma

**NIM:** 2301010666  
**Kelas:** XX  
**Mata Kuliah:** Machine Learning  
**Dosen:** Dr. I Gede Totok Suryawan, S.Kom., M.T.

**Dataset:** Hotel Booking Demand  
**Jalur:** A – Klasifikasi  
**Target:** is_canceled  
**Algoritma:** KNN dan DummyClassifier  
**Random State:** 666

### 1. Pemahaman Masalah dan Data

#### A. Latar Belakang

Pemesanan hotel merupakan salah satu aktivitas penting dalam industri pariwisata. Salah satu permasalahan yang sering dihadapi pihak hotel adalah pembatalan pemesanan oleh pelanggan. Pembatalan dapat memengaruhi tingkat okupansi kamar dan perencanaan operasional hotel.

Oleh karena itu, diperlukan analisis menggunakan Machine Learning untuk memprediksi kemungkinan pembatalan pemesanan hotel berdasarkan karakteristik pemesanan pelanggan.

#### B. Tujuan Analisis

Tujuan penelitian ini adalah membangun model klasifikasi menggunakan algoritma K-Nearest Neighbors (KNN) untuk memprediksi apakah suatu pemesanan hotel akan dibatalkan atau tidak.

#### C. Pengguna Hasil Analisis

Hasil analisis dapat digunakan oleh pihak manajemen hotel, bagian reservasi, dan bagian operasional untuk membantu memahami pola pembatalan pemesanan.

#### D. Keputusan yang Dibantu

Hasil prediksi diharapkan membantu pihak hotel mengidentifikasi pemesanan yang berisiko dibatalkan sehingga dapat mempertimbangkan strategi konfirmasi reservasi dan pengelolaan kamar.

#### E. Sumber Dataset

Dataset yang digunakan adalah Hotel Booking Demand yang diperoleh dari Kaggle.

Link: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Tanggal akses: 25 November 2026.

Target penelitian adalah kolom `is_canceled`, dengan nilai 0 berarti pemesanan tidak dibatalkan dan nilai 1 berarti pemesanan dibatalkan.

#### F. Lisensi dan Privasi

Dataset berasal dari penelitian Hotel Booking Demand Datasets oleh Antonio, Almeida, dan Nunes (2019). Identitas hotel dan pelanggan tidak ditampilkan dalam dataset. Lisensi artikel sumber adalah Creative Commons Attribution 4.0 (CC BY 4.0). Ketentuan distribusi salinan Kaggle tetap perlu diperiksa pada halaman sumber sebelum dataset disebarkan kembali.

In [4]:

# ==========================================
# 1. IMPORT LIBRARY DAN MEMBACA DATASET
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 666

# Membaca dataset
df = pd.read_csv('/content/hotel_bookings.csv')

# Menampilkan lima data pertama
display(df.head())

print("Jumlah baris dan kolom:", df.shape)
print("\nInformasi dataset:")
df.info()

print("\nJumlah missing value:")
print(df.isnull().sum())

print("\nJumlah data duplikat:")
print(df.duplicated().sum())

print("\nDistribusi target:")
print(df['is_canceled'].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: '/content/hotel_bookings.csv'

#### Keputusan saya

Saya memilih dataset Hotel Booking Demand karena memiliki jumlah data dan fitur yang memenuhi persyaratan UTS Machine Learning. Penelitian ini menggunakan jalur klasifikasi karena target `is_canceled` memiliki dua kategori, yaitu 0 dan 1.

Saya menggunakan algoritma KNN sebagai model utama dan DummyClassifier sebagai baseline. Random state ditetapkan sebesar 666 sesuai tiga digit terakhir NIM saya agar proses pembagian data dan pemodelan dapat direproduksi.

In [ ]:

# Grafik 1: Distribusi pembatalan hotel

plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x='is_canceled'
)

plt.title('Distribusi Pembatalan Pemesanan Hotel')
plt.xlabel('Status Pembatalan (0=Tidak, 1=Ya)')
plt.ylabel('Jumlah Pemesanan')
plt.xticks([0,1], ['Tidak Dibatalkan', 'Dibatalkan'])

plt.show()


In [ ]:
# Grafik 1: distribusi target
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="is_canceled")
plt.title("Distribusi Status Pembatalan Pemesanan")
plt.xlabel("is_canceled (0 = tidak batal, 1 = batal)")
plt.ylabel("Jumlah pemesanan")
plt.show()

target_pct = df["is_canceled"].value_counts(normalize=True).mul(100).round(2)
print("Persentase target (%):")
display(target_pct)

In [ ]:

# Grafik 2: Distribusi jenis hotel

plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x='hotel',
    hue='hotel'
)

plt.title('Distribusi Jenis Hotel')
plt.xlabel('Jenis Hotel')
plt.ylabel('Jumlah Pemesanan')
plt.legend([], [], frameon=False)

plt.show()


In [ ]:

# Grafik 3: Lead time dan status pembatalan

plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='is_canceled',
    y='lead_time'
)

plt.title('Hubungan Lead Time dengan Pembatalan')
plt.xlabel('Status Pembatalan')
plt.ylabel('Lead Time (Hari)')
plt.xticks([0,1], ['Tidak Dibatalkan', 'Dibatalkan'])

plt.show()


### Interpretasi grafik
1. **Grafik distribusi target:** bandingkan jumlah kelas 0 dan 1. Jika jumlahnya berbeda jauh, data tidak seimbang; karena itu evaluasi tidak cukup hanya menggunakan akurasi, tetapi juga precision, recall, dan F1.
2. **Grafik lead time:** histogram memperlihatkan sebaran jarak hari pemesanan hingga kedatangan. Bentuk distribusi dan nilai ekstrem perlu diperhatikan saat pra-pemrosesan.
3. **Grafik jenis hotel:** tinggi batang menunjukkan proporsi pembatalan pada masing-masing jenis hotel. Perbedaan proporsi menunjukkan bahwa jenis hotel mungkin berkaitan dengan pembatalan, tetapi tidak membuktikan sebab-akibat.

### Keputusan saya
Saya menggunakan tiga grafik yang sesuai dengan Jalur A: distribusi target, distribusi salah satu fitur numerik, dan hubungan fitur dengan target. Interpretasi akhir harus disesuaikan dengan grafik yang benar-benar dihasilkan saat notebook dijalankan; jangan menyalin angka yang belum muncul dari data Anda.

## 3. Pra-pemrosesan Data

Langkah yang dilakukan:
- Menghapus baris duplikat yang benar-benar sama.
- Mengubah nilai `children` yang kosong menjadi median melalui imputasi di dalam pipeline.
- Menangani nilai numerik kosong dengan median dan kategori kosong dengan modus.
- Memeriksa outlier menggunakan IQR. Nilai ekstrem tidak otomatis dihapus karena beberapa nilai besar (misalnya lead time) mungkin merupakan pemesanan yang sah. Pada pipeline, fitur numerik di-*clip* berdasarkan batas IQR yang dipelajari hanya dari data latih pada setiap fold.
- Mengubah fitur kategori menjadi angka menggunakan One-Hot Encoding.
- Melakukan standardisasi fitur numerik karena KNN menggunakan jarak.
- Membagi data latih dan uji sebelum proses `fit` preprocessing; menggunakan `stratify=y`.

`SEED` wajib diganti dengan tiga digit terakhir NIM Anda.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

SEED = 666  # Tiga digit terakhir NIM: 666

# Hilangkan duplikat persis
df = df.drop_duplicates().copy()

# Kolom target dan fitur
target = "is_canceled"
fitur = [
    "hotel", "lead_time", "arrival_date_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "meal", "market_segment",
    "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "reserved_room_type",
    "booking_changes", "deposit_type", "customer_type",
    "adr", "required_car_parking_spaces",
    "total_of_special_requests"
]

data_model = df[fitur + [target]].copy()
data_model = data_model.dropna(subset=[target])
X = data_model[fitur]
y = data_model[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Baris setelah hapus duplikat:", len(df))
print("Data latih:", X_train.shape, "Data uji:", X_test.shape)
print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

In [ ]:
# Transformer clipping outlier berbasis IQR.
# Batas dipelajari hanya dari data yang sedang di-fit (misalnya data latih/fold CV).
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.q1_ = np.nanpercentile(X, 25, axis=0)
        self.q3_ = np.nanpercentile(X, 75, axis=0)
        iqr = self.q3_ - self.q1_
        self.lower_ = self.q1_ - self.factor * iqr
        self.upper_ = self.q3_ + self.factor * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)

# Ringkasan jumlah outlier IQR pada data latih (untuk pelaporan/inspeksi)
num_train = X_train[numeric_features].copy()
q1 = num_train.quantile(0.25)
q3 = num_train.quantile(0.75)
iqr = q3 - q1
bawah = q1 - 1.5 * iqr
atas = q3 + 1.5 * iqr
jumlah_outlier = ((num_train < bawah) | (num_train > atas)).sum()
display(pd.DataFrame({
    "Q1": q1, "Q3": q3, "IQR": iqr,
    "Batas bawah": bawah, "Batas atas": atas,
    "Jumlah outlier data latih": jumlah_outlier
}))

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_iqr", IQRClipper(factor=1.5)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Pra-pemrosesan disiapkan dalam ColumnTransformer.")

### Keputusan saya
Saya menghapus duplikat persis agar catatan yang sama tidak dihitung berulang. Missing value ditangani dengan median untuk numerik dan modus untuk kategori. Outlier diperiksa memakai IQR dan di-*clip* dengan batas yang dipelajari dari data latih/fold, bukan dari seluruh dataset. One-Hot Encoding digunakan untuk fitur kategori, sedangkan StandardScaler digunakan agar skala fitur numerik tidak mendominasi jarak KNN. Seluruh proses *fit* preprocessing berada di dalam Pipeline untuk mencegah data leakage.

## 4. Pemodelan

Sesuai soal Jalur A, model utama yang digunakan adalah **K-Nearest Neighbors (KNN)** dan dibandingkan dengan **DummyClassifier** sebagai baseline. Nilai k dipilih dari beberapa kandidat menggunakan 5-fold Stratified Cross-Validation pada data latih. `random_state` menggunakan SEED.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
k_values = [3, 5, 7, 9, 11, 15]
hasil_cv = []

for k in k_values:
    model_knn_cv = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(
        model_knn_cv, X_train, y_train,
        cv=cv, scoring="f1", n_jobs=-1
    )
    hasil_cv.append({
        "k": k,
        "F1 rata-rata": scores.mean(),
        "F1 standar deviasi": scores.std()
    })

tabel_cv = pd.DataFrame(hasil_cv).sort_values("F1 rata-rata", ascending=False)
display(tabel_cv)
best_k = int(tabel_cv.iloc[0]["k"])
print("Nilai k terbaik berdasarkan rata-rata F1 CV:", best_k)

In [ ]:
plt.figure(figsize=(7,4))
plt.errorbar(
    tabel_cv["k"], tabel_cv["F1 rata-rata"],
    yerr=tabel_cv["F1 standar deviasi"], marker="o", capsize=4
)
plt.title("Perbandingan F1 5-Fold Cross-Validation untuk KNN")
plt.xlabel("Nilai k")
plt.ylabel("F1 rata-rata (± standar deviasi)")
plt.xticks(k_values)
plt.grid(True)
plt.show()

In [ ]:
knn_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=best_k))
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

# Fit hanya pada data latih
knn_model.fit(X_train, y_train)
baseline_model.fit(X_train, y_train)

### Keputusan saya
KNN dipilih karena sesuai untuk klasifikasi dan dapat memanfaatkan kedekatan pola antar pemesanan setelah fitur numerik diskalakan dan fitur kategori di-encode. Nilai k dipilih dari hasil rata-rata F1 5-fold cross-validation, bukan ditentukan berdasarkan tebakan. DummyClassifier menjadi pembanding sederhana untuk mengetahui apakah KNN benar-benar memberi peningkatan.

## 5. Evaluasi dan Interpretasi

Model diuji satu kali pada data uji yang tidak digunakan saat pelatihan. Metrik yang ditampilkan: confusion matrix, accuracy, precision, recall, dan F1. Perhitungan manual minimal satu metrik dilakukan dari confusion matrix.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

def evaluasi(model, X_eval, y_eval, nama):
    pred = model.predict(X_eval)
    cm = confusion_matrix(y_eval, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    hasil = {
        "Model": nama,
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1": f1_score(y_eval, pred, zero_division=0),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }
    return hasil, pred, cm

hasil_knn, pred_knn, cm_knn = evaluasi(knn_model, X_test, y_test, "KNN")
hasil_dummy, pred_dummy, cm_dummy = evaluasi(baseline_model, X_test, y_test, "DummyClassifier")

tabel_evaluasi = pd.DataFrame([hasil_knn, hasil_dummy])
display(tabel_evaluasi)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_knn,
    display_labels=["Tidak batal (0)", "Batal (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – KNN")
plt.show()

tn, fp, fn, tp = cm_knn.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
if (tp + fp) > 0:
    precision_manual = tp / (tp + fp)
    print(f"Perhitungan manual Precision = TP / (TP + FP) = {tp} / ({tp} + {fp}) = {precision_manual:.4f}")
else:
    print("Precision tidak dapat dihitung karena TP + FP = 0.")

In [ ]:
# Bandingkan skor train dan test untuk indikasi overfitting
pred_train = knn_model.predict(X_train)
pred_test = knn_model.predict(X_test)

perbandingan_train_test = pd.DataFrame([
    {"Data": "Latih", "Accuracy": accuracy_score(y_train, pred_train),
     "F1": f1_score(y_train, pred_train, zero_division=0)},
    {"Data": "Uji", "Accuracy": accuracy_score(y_test, pred_test),
     "F1": f1_score(y_test, pred_test, zero_division=0)}
])
display(perbandingan_train_test)

print("Classification report KNN:")
print(classification_report(y_test, pred_test, zero_division=0))

### Cara membaca hasil
- **Accuracy:** proporsi seluruh prediksi yang benar.
- **Precision kelas 1:** dari seluruh pemesanan yang diprediksi batal, berapa yang benar-benar batal.
- **Recall kelas 1:** dari seluruh pemesanan yang benar-benar batal, berapa yang berhasil ditemukan model.
- **F1:** rata-rata harmonik precision dan recall.
- **TN:** tidak batal dan diprediksi tidak batal.
- **FP:** tidak batal tetapi diprediksi batal.
- **FN:** batal tetapi diprediksi tidak batal.
- **TP:** batal dan diprediksi batal.

Jika skor data latih jauh lebih tinggi daripada data uji, ada indikasi overfitting. Bandingkan pula KNN dengan DummyClassifier. Model utama baru layak dipertimbangkan jika metrik yang relevan lebih baik daripada baseline dan hasilnya masuk akal untuk kebutuhan operasional.

### Keputusan saya
Saya mengevaluasi KNN dan baseline pada data uji yang sama. Precision dan recall kelas pembatalan penting karena kesalahan prediksi dapat memengaruhi keputusan pengelola hotel. Kesimpulan akhir harus menggunakan angka aktual dari tabel evaluasi setelah notebook dijalankan.

## 6. Kesimpulan

**Isi bagian ini setelah seluruh sel berhasil dijalankan.** Jangan mengarang angka. Salin nilai dari tabel evaluasi dan tabel perbandingan train-test.

Contoh format kesimpulan yang harus dilengkapi:
1. Model yang dipilih adalah **[KNN/DummyClassifier]** karena nilai F1 pada data uji sebesar **[isi angka]**, dibandingkan baseline sebesar **[isi angka]**.
2. Nilai recall kelas pembatalan sebesar **[isi angka]**, artinya model berhasil menemukan sekitar **[isi persentase]%** dari seluruh pemesanan yang benar-benar dibatalkan.
3. Selisih skor F1 data latih **[isi]** dan data uji **[isi]** menunjukkan **[indikasi overfitting/tidak terlihat overfitting yang besar]**.
4. Rekomendasi: pihak reservasi dapat memprioritaskan pemeriksaan manual atau konfirmasi untuk pemesanan yang diprediksi berisiko batal. Jangan membatalkan pemesanan secara otomatis hanya berdasarkan prediksi.
5. Keterbatasan: data berasal dari hotel di Portugal pada 2015–2017, sehingga pola mungkin berbeda dari kondisi hotel saat ini atau hotel di Indonesia. Data historis juga tidak menjamin perilaku masa depan.
6. Saran: gunakan data hotel lokal yang lebih baru, evaluasi berkala, bandingkan algoritma kedua (misalnya SVM), dan pertimbangkan biaya salah prediksi sebelum digunakan operasional.

### Keputusan saya
Saya akan menyatakan model layak dipertimbangkan hanya jika hasil uji mengungguli baseline pada metrik yang relevan, khususnya F1/recall kelas pembatalan, serta tidak menunjukkan overfitting yang berat. Jika tidak, model belum layak digunakan dan perlu perbaikan.

## Persiapan Repository GitHub dan Pernyataan Penggunaan AI

Struktur repository sesuai instruksi UTS:
```text
UTS-ML-NIM-Nama/
├── README.md
├── notebook/
│   └── UTS_ML_NIM_Nama.ipynb
├── data/
│   └── hotel_bookings.csv
└── reports/
    └── (grafik utama dan slide presentasi jika ada)
```

### Template Pernyataan Penggunaan AI untuk README
> Saya menggunakan ChatGPT untuk membantu memahami instruksi UTS, menyusun kerangka notebook, menjelaskan tahapan CRISP-DM, dan membantu penulisan contoh kode Python. Saya memeriksa hasil dengan menjalankan seluruh sel notebook, memeriksa sumber dan struktur dataset, serta meninjau kembali output grafik dan metrik. Saya memahami dan dapat menjelaskan serta mengubah kode yang digunakan.

**Penting:** sesuaikan pernyataan tersebut dengan penggunaan AI yang benar-benar Anda lakukan. Dosen meminta Anda mampu menjelaskan dan mengubah setiap bagian kode saat presentasi.

### Checklist sebelum dikumpulkan
- [ ] Dataset sudah diregistrasi di ELSA dan belum dipakai mahasiswa lain.
- [ ] `SEED` sudah diganti dengan tiga digit terakhir NIM.
- [ ] File CSV berasal dari sumber yang dicantumkan.
- [ ] Semua sel dijalankan melalui Runtime → Run all dan output terlihat.
- [ ] Interpretasi dan kesimpulan sudah dilengkapi berdasarkan angka hasil sendiri.
- [ ] README berisi identitas, sumber/lisensi data, dan pernyataan penggunaan AI.
- [ ] Repository GitHub publik mengikuti nama `UTS-ML-NIM-Nama`.
- [ ] Link repository dikirim ke ELSA sebelum tenggat pada soal.